# 03 — Podela na trening i test skup

Ovo je verovatno najvažniji korak u celom projektu: podela podataka na trening (train) i test skup.

Vazan je zato što sve što radimo dalje (detekcija outliera, imputacija, feature engineering, skaliranje) mora da se oslanja samo na trening skup. Test skup diramo tek jednom, na samom kraju, za finalnu evaluaciju.

Time sprečavamo curenje informacija (**data leakage**) i obezbeđujemo da evaluacija bude poštena — da stvarno pokaže kako će se model ponašati na podacima koje ranije nije video.

## 1. Učitavanje podataka i strukturne popravke

Učitavamo skup i radimo dve strukturne popravke koje smo najavili ranije:

1. Konverzija `TotalCharges` u broj — pošto je 11 vrednosti sadržalo prazan razmak, cela kolona je bila učitana kao tekst.
2. Uklanjanje tih 11 redova sa `NaN` u `TotalCharges` — reč je o korisnicima sa `tenure = 0` (tek pristigli, još nenaplaćeni). Ima ih zanemarljivo malo (0.16%), a njihova specifičnost bi mogla da unese šum.

Napomena: ove popravke su strukturne — ne zavise od raspodele podataka nego su posledica objektivnih problema u sirovom fajlu. Zato ih radimo pre split-a, bez rizika od data leakage-a.

In [ ]:
import pandas as pd
import numpy as np

# train_test_split je funkcija za podelu podataka na trening i test skup.
from sklearn.model_selection import train_test_split


pd.set_option("display.max_columns", None)

df = pd.read_csv("../WA_Fn-UseC_-Telco-Customer-Churn.csv")


df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Strukturna popravka 2: brisanje 11 redova sa NaN u TotalCharges.
# Ovo su korisnici sa tenure=0 (novi, još nisu naplaćeni).
df = df.dropna(subset=["TotalCharges"])

# .reset_index(drop=True) resetuje indekse (redne brojeve) redova.
df = df.reset_index(drop=True)

print(f"Nakon strukturnih popravki: {df.shape[0]} redova × {df.shape[1]} kolona")

Nakon strukturnih popravki: 7032 redova × 21 kolona


## 2. Razdvajanje atributa (X) od ciljne promenljive (y)

Pre podele na train/test razdvajamo dve stvari:
- `X` — atributi (features) na osnovu kojih model uči i predviđa
- `y` — ciljna promenljiva (target) koju predviđamo, tj. `Churn`

Uz to izbacujemo `customerID`, pošto je to samo jedinstveni identifikator bez prediktivne vrednosti — svaki korisnik ima svoj ID, pa tu nema obrasca koji bi model mogao da nauči.

Oko oznaka: veliko `X` se po konvenciji koristi za matricu atributa (više kolona), a malo `y` za vektor ciljne promenljive (jedna kolona).

In [10]:
# Definišemo ciljnu promenljivu (y) — kolonu koju predviđamo.
y = df["Churn"]

# Definišemo matricu atributa (X) — sve kolone OSIM Churn-a i customerID-a.
# df.drop(columns=[...]) vraća novi DataFrame bez tih kolona.
# customerID izbacujemo jer nema prediktivnu vrednost.
# Churn izbacujemo jer je to ciljna promenljiva (ne sme biti među atributima).
X = df.drop(columns=["customerID", "Churn"])

# Proveravamo dimenzije.
print(f"X (atributi): {X.shape[0]} redova × {X.shape[1]} kolona")
print(f"y (ciljna promenljiva): {y.shape[0]} vrednosti")
print(f"\nAtributi u X: {list(X.columns)}")

X (atributi): 7032 redova × 19 kolona
y (ciljna promenljiva): 7032 vrednosti

Atributi u X: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


## 3. Podela na trening i test skup

Sada delimo podatke na dva skupa:
- Trening (80%) — na njemu radimo sve dalje analize, transformacije i treniranje modela.
- Test (20%) — zaključavamo ga do kraja i koristimo samo za finalnu evaluaciju.

Parametri koje prosleđujemo:

- `test_size=0.2` — 20% podataka ide u test skup.
- `stratify=y` — stratifikacija po ciljnoj promenljivoj, da proporcija klasa (Yes/No) bude ista u train i test skupu. Kod nebalansiranog skupa kao što je naš (~73% : ~27%) ovo je bitno.
- `random_state=42` — fiksira „seme" slučajnog procesa, pa svako pokretanje daje istu podelu. Bez toga nema ponovljivosti eksperimenata.
- `shuffle=True` — redovi se prvo izmešaju, da eventualni redosled u sirovim podacima ne utiče na podelu (npr. da svi noviji korisnici ne završe na kraju).

In [11]:
# Delimo X i y na train i test skup jednim pozivom funkcije.
# train_test_split vraća 4 objekta u fiksnom redosledu:
#   X_train, X_test, y_train, y_test
# Redosled je uvek ovakav — X pre y, train pre test.
X_train, X_test, y_train, y_test = train_test_split(
    X,                    # matrica atributa
    y,                    # ciljna promenljiva
    test_size=0.2,        # 20% za test
    stratify=y,           # stratifikacija po ciljnoj promenljivoj
    random_state=42,      # fiksiramo slučajnost za ponovljivost
    shuffle=True          # izmešaj redove pre podele
)

# Prikazujemo dimenzije oba skupa da potvrdimo podelu.
print("Dimenzije nakon podele:")
print(f"  X_train: {X_train.shape[0]} redova x {X_train.shape[1]} kolona")
print(f"  X_test:  {X_test.shape[0]} redova x {X_test.shape[1]} kolona")
print(f"  y_train: {y_train.shape[0]} vrednosti")
print(f"  y_test:  {y_test.shape[0]} vrednosti")

# Procentualno odnos
ukupno = X_train.shape[0] + X_test.shape[0]
print(f"\nProcenat train: {100 * X_train.shape[0] / ukupno:.2f}%")
print(f"Procenat test:  {100 * X_test.shape[0] / ukupno:.2f}%")

Dimenzije nakon podele:
  X_train: 5625 redova x 19 kolona
  X_test:  1407 redova x 19 kolona
  y_train: 5625 vrednosti
  y_test:  1407 vrednosti

Procenat train: 79.99%
Procenat test:  20.01%


## 4. Provera stratifikacije

Posle podele proveravamo da li je stratifikacija stvarno odradila posao — da proporcija klasa `Churn` bude ista (ili skoro ista) u train i test skupu.

Ako se procenti bitno razlikuju, nešto ne valja i treba videti šta.

In [12]:
# Procentualna raspodela Churn u originalnom skupu, train skupu i test skupu.
# normalize=True vraća proporcije umesto apsolutnih brojeva.
print("Proporcije Churn klasa:")
print("\nOriginalni skup:")
print(y.value_counts(normalize=True) * 100)

print("\nTrain skup:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest skup:")
print(y_test.value_counts(normalize=True) * 100)

Proporcije Churn klasa:

Originalni skup:
Churn
No     73.421502
Yes    26.578498
Name: proportion, dtype: float64

Train skup:
Churn
No     73.422222
Yes    26.577778
Name: proportion, dtype: float64

Test skup:
Churn
No     73.418621
Yes    26.581379
Name: proportion, dtype: float64


### Rezultat provere

Proporcije klase `Churn` su skoro identične u sva tri skupa (originalni, train, test) — svuda odnos oko 73.4% : 26.6%. Stratifikacija je, dakle, prošla kako treba i oba skupa su reprezentativna po ciljnoj promenljivoj.

Praktično, to znači da će evaluacija na test skupu biti poštena — odnos klasa koji model vidi pri evaluaciji odgovara stvarnom odnosu u populaciji.

## 5. Čuvanje train i test skupova u fajlove

Da ne bi svaki put iznova radili učitavanje, konverziju i split, sačuvaćemo trenutno stanje train i test skupova u `data/processed/`.

Za format biramo CSV — čitljiv je, univerzalan i lako se otvara u bilo čemu (Excel, Python, R). Moglo bi i Parquet (brži i manji), ali za naših 7.032 reda CSV je sasvim dovoljan.

Čuvamo četiri fajla:
- `X_train.csv` i `y_train.csv` — trening podaci
- `X_test.csv` i `y_test.csv` — test podaci (ne diramo ih do kraja)

In [ ]:
putanja = "../data/processed/"

# index=False znači: ne čuvaj indekse (redne brojeve) kao dodatnu kolonu u CSV-u.
# Ako bismo ostavili default (index=True), pri sledećem učitavanju bismo dobili
# suvišnu kolonu "Unnamed: 0" — svima na nervima i beskorisnu.
X_train.to_csv(putanja + "X_train.csv", index=False)
X_test.to_csv(putanja + "X_test.csv", index=False)
y_train.to_csv(putanja + "y_train.csv", index=False)
y_test.to_csv(putanja + "y_test.csv", index=False)

print("Fajlovi uspešno sačuvani u folder 'data/processed/':")
print("  - X_train.csv")
print("  - X_test.csv")
print("  - y_train.csv")
print("  - y_test.csv")

Fajlovi uspešno sačuvani u folder 'data/processed/':
  - X_train.csv
  - X_test.csv
  - y_train.csv
  - y_test.csv


## 6. Verifikacija: učitavanje fajlova nazad

Kao završna provera, učitavamo sačuvane fajlove nazad i uveravamo se da
su dimenzije očuvane. Ovo je važno jer sledeće sveske startuju baš iz
ovih fajlova

In [17]:
# Učitavamo fajlove nazad da proverimo da je čuvanje bilo korektno.
X_train_check = pd.read_csv(putanja + "X_train.csv")
X_test_check = pd.read_csv(putanja + "X_test.csv")
y_train_check = pd.read_csv(putanja + "y_train.csv")
y_test_check = pd.read_csv(putanja + "y_test.csv")

print("Dimenzije nakon učitavanja iz fajlova:")
print(f"  X_train: {X_train_check.shape[0]} × {X_train_check.shape[1]}")
print(f"  X_test:  {X_test_check.shape[0]} × {X_test_check.shape[1]}")
print(f"  y_train: {y_train_check.shape[0]}")
print(f"  y_test:  {y_test_check.shape[0]}")

# Provera identičnosti — dimenzije treba da se poklope sa originalnim.
provere = [
    X_train.shape == X_train_check.shape,
    X_test.shape == X_test_check.shape,
    y_train.shape[0] == y_train_check.shape[0],
    y_test.shape[0] == y_test_check.shape[0],
]

if all(provere):
    print("\nSve dimenzije se poklapaju. Fajlovi su korektno sačuvani.")
else:
    print("\nUPOZORENJE: dimenzije se ne poklapaju — istražiti problem!")

Dimenzije nakon učitavanja iz fajlova:
  X_train: 5625 × 19
  X_test:  1407 × 19
  y_train: 5625
  y_test:  1407

Sve dimenzije se poklapaju. Fajlovi su korektno sačuvani.


---

## 7. Zaključak

U ovoj svesci smo odradili ono što je verovatno najvažniji korak u projektu — podelu podataka na trening i test skup.

Ukratko, šta je urađeno:

1. Strukturne popravke — konverzija `TotalCharges` u broj i uklanjanje 11 redova sa nedostajućim vrednostima.
2. Razdvajanje `X` i `y` — atributi (19 kolona) odvojeni od ciljne promenljive (`Churn`).
3. Stratifikovana podela — 80% train, 20% test, uz očuvanje proporcija klasa u oba skupa.
4. Provera — proporcije `Churn` su praktično iste u train i test skupu (~73.4% / ~26.6%).
5. Čuvanje — sva četiri objekta (`X_train`, `X_test`, `y_train`, `y_test`) sačuvana u `data/processed/`.

Dimenzije posle podele: train ima 5.625 korisnika (80%), test 1.407 (20%).

Princip kojeg se od sada držimo: sve odluke, analize i transformacije rade se samo na trening skupu. Test skup ne otvaramo, ne analiziramo i ne koristimo ni za kakvo računanje statistika sve do kraja projekta, kada služi za finalnu evaluaciju modela.